In [1]:
# These things will probably be moved to a separate .py file so I can use the functions, but the notebook is good for visualization and rapid iteration.
# Goal: generate 20 "good" test subjects and save them to file.

In [23]:
import os
import json
import numpy as np
import rescomp as rc
from rescomp import optimizer as rcopt
from scipy import sparse
import networkx as nx
from utils import get_components, get_response
from rescomp.optimizer.optimizer_functions import get_vptime

Test for generating and saving networks

In [26]:
# Setup
system_name = "lorenz"
system = rcopt.get_system(system_name)
hyper_parameters = rc.SYSTEMS[system_name]["rcomp_params"]
target_count = 3
vpt_threshold = 3.0
subjects = []
os.makedirs("subjects", exist_ok=True)

In [27]:
while len(subjects) < target_count:
    # Train reservoir with default hyperparameters
    tr, Utr, ts, Uts = rc.train_test_orbit(system_name, duration=1000, trainper=980/1000)
    res = rc.ResComp(**hyper_parameters)
    res.train(tr, Utr)

    # Evaluate VPT
    pre = res.predict(ts, r0=res.r0)
    vpt = get_vptime(system, ts, Uts, pre, vpttol=0.5)
    print(f"VPT: {vpt:.3f}")

    if vpt < vpt_threshold:
        continue

    # Extract network and components
    A = sparse.coo_matrix(res.res)
    G = nx.DiGraph(A)
    S = get_components(A)

    # Build component data keyed by index
    response_scaled = get_response(res, ts, Uts)
    components = {}
    for i, comp in enumerate(S):
        mask = np.array(comp.nodes)
        components[i] = {
            "size": len(comp.nodes),
            "nodes": mask.tolist(),
            "mean_response_t0": float(np.mean(response_scaled[mask, 0])),
        }

    idx = len(subjects)
    subjects.append({
        "vpt": vpt,
        "A": A,
        "G": G,
        "components": components,
    })

    # Save to disk
    sparse.save_npz(f"subjects/network_{idx:03d}.npz", A)
    meta = {
        "vpt": vpt,
        "system": system_name,
        "components": components,
    }
    with open(f"subjects/network_{idx:03d}.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved subject {idx + 1}/{target_count}")

print("Done.")

VPT: 1.020
VPT: 1.100
VPT: 1.830
VPT: 2.220
VPT: 0.900
VPT: 3.000
Saved subject 1/3
VPT: 0.430
VPT: 0.830
VPT: 0.520
VPT: 0.010
VPT: 1.300
VPT: 0.000
VPT: 0.800
VPT: 1.180
VPT: 0.330
VPT: 0.440
VPT: 2.270
VPT: 0.190
VPT: 0.480
VPT: 1.660
VPT: 0.510
VPT: 0.290
VPT: 0.890
VPT: 1.100
VPT: 0.530
VPT: 2.200
VPT: 2.070
VPT: 1.120
VPT: 3.020
Saved subject 2/3
VPT: 1.600
VPT: 0.310
VPT: 2.280
VPT: 0.090
VPT: 0.710
VPT: 0.940
VPT: 0.270
VPT: 1.470
VPT: 0.240
VPT: 1.750
VPT: 0.630
VPT: 0.190
VPT: 2.200
VPT: 1.020
VPT: 0.160
VPT: 0.050
VPT: 2.000
VPT: 0.600
VPT: 0.410
VPT: 0.280
VPT: 0.000
VPT: 0.480
VPT: 0.450
VPT: 2.320
VPT: 1.850
VPT: 1.470
VPT: 0.440
VPT: 1.310
VPT: 2.020
VPT: 0.820
VPT: 0.450
VPT: 0.800
VPT: 0.420
VPT: 0.890
VPT: 0.010
VPT: 0.850
VPT: 0.540
VPT: 1.230
VPT: 0.710
VPT: 0.730
VPT: 2.320
VPT: 2.110
VPT: 1.720
VPT: 0.470
VPT: 0.360
VPT: 1.670
VPT: 0.620
VPT: 0.470
VPT: 0.280
VPT: 1.960
VPT: 0.430
VPT: 1.570
VPT: 0.100
VPT: 0.110
VPT: 0.980
VPT: 0.140
VPT: 1.010
VPT: 0.160
VPT: 1.

In [28]:
A_loaded = sparse.load_npz("subjects/network_000.npz")
G_loaded = nx.DiGraph(A_loaded)
with open("subjects/network_000.json") as f:
    meta = json.load(f)
print(f"VPT: {meta['vpt']:.3f}")
print(f"Components: {len(meta['components'])}")

VPT: 3.000
Components: 789
